# Renewables in the UK: capacity, geography and the role of policy

**DSM050 Data Visualisation - Midterm coursework**

In recent years the UK has rapidly increased the share of renewables in its energy mix. This notebook explores how the UK's renewable sector has grown and changed, with a focus on making the data easy to understand and explore. The analysis uses the GOV.UK **Renewable Energy Planning Database (REPD)**, a project-level record of renewable electricity projects (>150 kW) across the UK.

The investigation is organised around three research questions:

1. **RQ1 - Technology mix & growth.** How has the composition of the UK's renewable *capacity* evolved over time, and to what extent has wind (offshore vs onshore) driven that change relative to solar and other sources?
2. **RQ2 - Regional distribution.** How is renewable capacity distributed across UK regions, and how has that geography evolved over time?
3. **RQ3 - Government support.** How are the support schemes (CfD, RO, FiT) associated with the scale and technology mix of renewable projects?

> **Note on this notebook.** This is the reproducible analysis; the accompanying report carries the full discussion. Each code block is preceded by a short description of what it does. Figures are saved to `figures/` and also shown inline.

## 1 | Data and preprocessing

The data comes from the [REPD monthly extract](https://www.gov.uk/government/publications/renewable-energy-planning-database-monthly-extract) (Q1 2026 release). It covers project ownership, technology type, government support, capacity, location and key dates - a full picture of the UK's renewables landscape, past and present.

### 1.1 | Loading and filtering

*Load the raw REPD sheet and take a first look at its shape.*

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load the data from the Excel file
data_raw = pd.read_excel("data/REPD_publication_Q1_2026.xlsx", sheet_name="REPD")
print(data_raw.shape)
data_raw.head(3)

(14316, 53)


,Old Ref ID,Ref ID,Record Last Updated (dd/mm/yyyy),Operator (or Applicant),Site Name,Technology Type,Storage Type,Storage Co-location REPD Ref ID,Installed Capacity (MWelec),Share Community Scheme,...,Appeal Granted,Planning Permission Granted,Secretary of State - Intervened,Secretary of State - Refusal,Secretary of State - Granted,Planning Permission Expired,Under Construction,Operational,Heat Network Ref,Solar Site Area (sqm)
0,N00053B,1,2026-01-16,RWE npower,Aberthaw Power Station Biomass,Biomass (co-firing),NaN,NaN,35,NaN,...,NaT,2004-09-03,NaT,NaN,NaT,NaT,2006-05-01,2007-05-01 00:00:00,NaN,NaN
1,AA110,2,2017-11-20,Orsted (formerly Dong Energy) / Peel Energy,Hunterston - cofiring,Biomass (co-firing),NaN,NaN,170,NaN,...,NaT,NaT,NaT,NaN,NaT,NaT,NaT,NaN,NaN,NaN
2,B0730,3,2019-12-20,Scottish and Southern Energy (SSE),Ferrybridge Multifuel 2 (FM2),EfW Incineration,NaN,NaN,70,NaN,...,NaT,2015-10-28,NaT,NaN,NaT,2020-10-28,2016-09-01,2019-12-20 00:00:00,NaN,NaN


The dataset has ~14,300 rows and 53 columns. Many columns are relevant, but several are not. We first remove columns that are definitely irrelevant (old IDs, the last-modified date) and move `Ref ID` to the row index.

*Drop clearly-irrelevant columns, then drop the block of planning/legal-status columns that aren't needed for a study of what has actually been built.*

In [2]:
# remove some irrelevent columns
irrelevant_columns = ["Old Ref ID", "Record Last Updated (dd/mm/yyyy)"]
data_raw = data_raw.drop(columns=irrelevant_columns)

# move Ref ID to the row names
data_raw.set_index("Ref ID", inplace=True)

# most of these columns are about the planning/legal status of projects, which is
# not directly relevant to the current state and past growth of renewables.
# NOTE: we deliberately KEEP "Under Construction" and "Planning Permission  Granted"
# (both date columns, the latter with a double space in the raw Q1-2026 file) so that
# Pending projects, which have no Operational date, still get a usable fallback
# timestamp later on (the derived "Project Date" column in section 1.4).
to_remove = [
    "Heat Network Ref", "Planning Permission Expired",
    "Secretary of State - Granted", "Secretary of State - Refusal",
    "Secretary of State - Intervened",
    "Appeal Granted", "Appeal Refused", "Appeal Withdrawn", "Appeal Lodged",
    "Planning Permission Refused", "Planning Application Withdrawn",
    "Judicial Review", "Type of Secretary of State Intervention",
    "Secretary of State Reference", "Appeal Reference",
    "Planning Application Reference", "Planning Authority", "Post Code",
    "Address", "Are they re-applying (Old REPD Ref) ",
    "Are they re-applying (New REPD Ref)", "Development Status",
    "Mounting Type for Solar", "Storage Co-location REPD Ref ID",
    "Storage Type", "Offshore Wind Round"]
data = data_raw.drop(columns=to_remove)
print("after column trim:", data.shape)

after column trim: (14316, 24)


### 1.2 | Development status

`Development Status (short)` has many values, but we can collapse them into three groups: **Operational**, **Pending** and **Terminated**. A dictionary maps each raw value to a general category.

> When re-running on the Q1 2026 data I found a new status value, `Application Granted`, that my original mapping did not cover. It means a project has planning consent but is not yet built, so it maps to **Pending**. Catching this mattered - an unmapped status would otherwise sit outside all three groups.

In [3]:
dev_status_mapping = {
    "Operational": "Operational",
    "Application Withdrawn": "Pending",
    "Application Refused": "Terminated",
    "Abandoned": "Terminated",
    "Appeal Refused": "Terminated",
    "Revised": "Terminated",
    "Appeal Withdrawn": "Terminated",
    "Planning Permission Expired": "Terminated",
    "Under Construction": "Pending",
    "Decommissioned": "Terminated",
    "Awaiting Construction": "Pending",
    "No Application Required": "Operational",
    "Application Submitted": "Pending",
    "Appeal Lodged": "Pending",
    "Application submitted": "Pending",
    "Application Granted": "Pending"}  # new status in the Q1-2026 data

data["Development Status"] = data["Development Status (short)"].replace(dev_status_mapping)
data = data.drop(columns=["Development Status (short)"])
print("unmapped statuses:", set(data["Development Status"].unique()) - {"Operational", "Pending", "Terminated"})

unmapped statuses: set()


### 1.3 | Technology type

We only want renewable *generation* projects, so we combine similar technologies and remove pure storage. A few technology types can be combined (e.g. co-firing and dedicated biomass -> **Biomass**; large and small hydro -> **Hydro**), and storage methods (batteries, pumped hydro, etc.) are removed entirely.

In [4]:
tech_type_mapping = {
    "Biomass (co-firing)": "Biomass", "Biomass (dedicated)": "Biomass",
    "Large Hydro": "Hydro", "Small Hydro": "Hydro",
    "Tidal Stream": "Tidal", "Tidal Lagoon": "Tidal", "Shoreline Wave": "Tidal",
    "EfW Incineration": "Waste to Energy", "Advanced Conversion Technologies": "Other",
    "Anaerobic Digestion": "Waste to Energy", "Landfill Gas": "Waste to Energy",
    "Sewage Sludge Digestion": "Waste to Energy", "Hot Dry Rocks (HDR)": "Geothermal",
    "Geothermal": "Other", "Geothermal ": "Other", "Air Source Heat Pumps": "Other",
    "Unknown": "Other"}
data["Technology Type"] = data["Technology Type"].replace(tech_type_mapping)

storage_types = ["Battery", "Pumped Storage Hydroelectricity", "Liquid Air Energy Storage",
                 "Flywheels", "Fuel Cell (Hydrogen)", "Hydrogen", "Compressed Air Energy Storage"]
data = data[~data["Technology Type"].isin(storage_types)]
print("technologies:", sorted(data["Technology Type"].unique()))
print("shape:", data.shape)

technologies: ['Biomass', 'Geothermal', 'Hydro', 'Other', 'Solar Photovoltaics', 'Tidal', 'Waste to Energy', 'Wind Offshore', 'Wind Onshore']
shape: (11617, 24)


### 1.4 | Cleaning columns, types and missing values

This block does the bulk of the tidying: renaming columns to friendlier names, coercing numeric columns (filling non-applicable blanks with 0), parsing the operational date, and standardising text fields.

> Two more pieces of data drift surfaced in the Q1 2026 file: the CfD scheme added later allocation rounds (**AR7**, **AR7a**), and an `Investment Contract` value with different capitalisation. Both are handled below so no projects are silently lost. The single reallocated `AR6 & AR7a` project is folded into `AR7a`.

In [5]:
# Operator (or Applicant)
data.rename(columns={"Operator (or Applicant)": "Operator"}, inplace=True)
# operational date -> datetime
data["Operational"] = pd.to_datetime(data["Operational"], errors='coerce')
# two more date columns kept as datetime fallbacks (see section 1.1)
data["Under Construction"] = pd.to_datetime(data["Under Construction"], errors='coerce')
data["Planning Permission  Granted"] = pd.to_datetime(data["Planning Permission  Granted"], errors='coerce')
# a single derived timestamp: Operational, else Under Construction, else Planning Permission Granted.
# This gives Pending projects (no Operational date) a usable "when" for the RQ3 time-matching below,
# while leaving every existing use of `Operational` untouched.
data["Project Date"] = (data["Operational"]
                        .fillna(data["Under Construction"])
                        .fillna(data["Planning Permission  Granted"]))

# Installed Capacity: rename + coerce numeric + fill NA with 0
data.rename(columns={"Installed Capacity (MWelec)": "Installed Capacity (MW)"}, inplace=True)
data["Installed Capacity (MW)"] = pd.to_numeric(data["Installed Capacity (MW)"], errors='coerce').fillna(0)

data = data.drop(columns=["Share Community Scheme"], errors='ignore')

# CHP: map yes/no variants to bool
data.rename(columns={"CHP Enabled": "CHP"}, inplace=True)
data["CHP"] = data["CHP"].map({"Yes": True, "yes": True, "NO": False, "no": False, "No": False}).fillna(False)
data = data.drop(columns=["CHP Enabled"], errors='ignore')

# CfD Allocation Round: clean, fold the 1 reallocated project into AR7a, order as an ordinal category
cfd_order = ["None", "AR1", "AR2", "AR3", "AR4", "AR5", "AR6", "AR7a", "AR7"]
data["CfD Allocation Round"] = (
    data["CfD Allocation Round"]
    .replace({np.nan: "None", "nan": "None", "Investment contract": "None",
              "Investment Contract": "None", "AR6 & AR7a": "AR7a"})
    .str.strip().replace("", "None"))
data["CfD Allocation Round"] = pd.Categorical(data["CfD Allocation Round"], categories=cfd_order, ordered=True)

# remaining numeric columns: rename + coerce + fill 0
data.rename(columns={"RO Banding (ROC/MWh)": "RO Banding", "FiT Tariff (p/kWh)": "FiT Tariff",
                     "CfD Capacity (MW)": "CfD Capacity", "Turbine Capacity (MW)": "Turbine Capacity",
                     "Height of Turbines (m)": "Height of Turbine", "Solar Site Area (sqm)": "Solar Site Area"},
            inplace=True)
for c in ["RO Banding", "FiT Tariff", "CfD Capacity", "Turbine Capacity", "No. of Turbines",
          "Height of Turbine", "X-coordinate", "Y-coordinate", "Solar Site Area"]:
    data[c] = pd.to_numeric(data[c], errors='coerce').fillna(0)

# tidy text location fields
data["County"] = (data["County"].str.strip()
                  .str.replace(r"^Co\.?\s*", "", regex=True)
                  .str.replace(r"\s*/.*$", "", regex=True))
data["Country"] = data["Country"].str.strip()

Now we audit missing values. The `Operational` column is empty for projects that never became operational (pending or terminated) - those blanks are meaningful and we keep them. The handful of other missing values (a few location fields) are a tiny share of the data and can be dropped.

In [6]:
# remove rows missing any column EXCEPT the date columns, which are legitimately blank for
# non-operational / not-yet-started projects (Operational, the two fallback dates and Project Date).
date_cols = ["Operational", "Under Construction", "Planning Permission  Granted", "Project Date"]
data = data.dropna(subset=data.columns.difference(date_cols))
na = data.isna().sum()
print("remaining NAs (date columns are legitimately blank):\n", na[na > 0])
print("final shape:", data.shape)

remaining NAs (date columns are legitimately blank):
 Planning Permission  Granted    3329
Under Construction              9194
Operational                     8573
Project Date                    3260
dtype: int64
final shape: (11361, 24)


### 1.5 | Derived numeric frame and a note on dimensionality

Finally we build a min-max normalised copy of the numeric columns, kept separate so the main frame stays readable.

Most columns here are categorical, so classic **PCA** is a poor fit for dimensionality reduction. For the numeric columns PCA/t-SNE/UMAP are options, but the data is highly sparse (most rows are 0 for scheme-specific fields), so instead of a black-box projection this analysis manages dimensionality by **feature selection** - focusing each research question on the few variables that matter (capacity, technology, region, date, support scheme).

In [7]:
numeric_columns = data.select_dtypes(include=[np.number]).columns.tolist()
data_norm = (data[numeric_columns] - data[numeric_columns].min()) / (data[numeric_columns].max() - data[numeric_columns].min())
print("numeric columns:", numeric_columns)
data[numeric_columns].describe().T.round(2)

numeric columns: ['Installed Capacity (MW)', 'RO Banding', 'FiT Tariff', 'CfD Capacity', 'Turbine Capacity', 'No. of Turbines', 'Height of Turbine', 'X-coordinate', 'Y-coordinate', 'Solar Site Area']


,count,mean,std,min,25%,50%,75%,max
Installed Capacity (MW),11361.0,18.10,100.94,0.0,0.33,2.4,11.9,4100.00
RO Banding,11361.0,0.03,0.20,0.0,0.00,0.0,0.0,2.00
FiT Tariff,11361.0,0.07,0.67,0.0,0.00,0.0,0.0,10.96
CfD Capacity,11361.0,3.96,56.24,0.0,0.00,0.0,0.0,2400.00
Turbine Capacity,11361.0,0.54,2.11,0.0,0.00,0.0,0.0,125.00
No. of Turbines,11361.0,2.35,10.81,0.0,0.00,0.0,0.0,341.00
Height of Turbine,11361.0,12.18,41.98,0.0,0.00,0.0,0.0,403.00
X-coordinate,11361.0,391999.67,125356.84,0.0,314371.00,398494.0,481592.0,708047.00
Y-coordinate,11361.0,371647.31,219606.86,0.0,193159.00,322038.0,524549.0,1202671.00
Solar Site Area,11361.0,92094.32,498126.73,0.0,0.00,0.0,9100.0,18310000.00


## 2 | Exploratory data analysis

We start with an overview of the key variables. By measurement type these are: **nominal** - Technology Type, Country, Region; **ordinal** - CfD Allocation Round; **numerical** - Installed Capacity (MW), and the other continuous fields.

### 2.1 | Numerical variables

*Summary statistics and a scatter-matrix of the numeric columns.*

In [8]:
# A consistent technology colour map + row order, defined once here so every figure below
# (EDA counts, RQ1 pie/area, the RQ3 heatmap and the dashboard) shares the same encoding.
colors = {
    "Wind Offshore": "#2878B5", "Wind Onshore": "#74A9CF",
    "Solar Photovoltaics": "#F9D057", "Other": "#BEBEBE",
    "Biomass": "#43AA8B", "Hydro": "#577590", "Tidal": "#90BE6D",
    "Waste to Energy": "#F3722C", "Geothermal": "#BC5090"}
tech_types = ['Biomass', 'Waste to Energy', 'Hydro', 'Solar Photovoltaics', 'Tidal',
              'Wind Offshore', 'Wind Onshore', 'Geothermal', 'Other']

numeric_cols = data_norm.columns.tolist()
axarr = pd.plotting.scatter_matrix(data[numeric_cols], figsize=(15, 15), diagonal='kde', alpha=0.2)
plt.gcf().savefig("figures/eda_numeric_scatter_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\2714232759.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The numeric data is highly sparse - most values are 0, because scheme- and turbine-specific fields only apply to certain project types. We can see clear relationships among the turbine variables, and the X/Y coordinates trace out a little map of the UK.

### 2.2 | Categorical variables

*Counts for the key categorical variables.*

In [9]:
key_categorical_columns = ["Technology Type", "Development Status", "Country", "CfD Allocation Round"]
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.ravel()

# distinct qualitative colour per country bar (no existing country colour map)
country_palette = sns.color_palette("Set2", n_colors=data["Country"].nunique())

for i, col in enumerate(key_categorical_columns):
    ax = axes[i]
    vc = data[col].value_counts()
    if col == "CfD Allocation Round":
        # drop the 'None' bar: it swamps the panel and is the implicit message anyway
        vc = vc.drop("None", errors="ignore")
    vc = vc[vc > 0]
    cats = list(vc.index)
    if col == "Technology Type":
        bar_colors = [colors[c] for c in cats]
    elif col == "Country":
        bar_colors = [country_palette[j] for j in range(len(cats))]
    else:
        bar_colors = "#577590"
    ax.bar(range(len(cats)), vc.values, color=bar_colors, edgecolor="white")
    ax.set_xticks(range(len(cats)))
    ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=14)
    ax.set_title(col, fontsize=21)
    ax.set_ylabel("count", fontsize=16)
    ax.tick_params(axis="y", labelsize=14)

fig.tight_layout()
fig.savefig("figures/eda_categorical_counts.png", dpi=150, bbox_inches="tight")
plt.show()

# count of projects with no CfD allocation round (dropped from that panel) -> reported in the caption
_none_n = int((data["CfD Allocation Round"] == "None").sum()); _total_n = len(data)
print(f"CfD 'None' omitted from panel: n={_none_n} ({_none_n / _total_n * 100:.1f}%)")

CfD 'None' omitted from panel: n=10840 (95.4%)


C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\3499606041.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Solar is the most common technology by project count, followed by onshore wind. Most projects are in England, then Scotland. The vast majority of projects sit outside the CfD scheme (`None`).

## 3 | RQ1 - Technology mix and growth over time

We restrict to **operational** projects and look at the current share of capacity, then how the mix has evolved.

*Current share of installed capacity by technology (pie).*

In [10]:
operational_data = data[data["Development Status"] == "Operational"].copy()
operational_data["Year"] = operational_data["Operational"].dt.year

capacity_by_tech = operational_data.groupby("Technology Type")["Installed Capacity (MW)"].sum().sort_values(ascending=False)
fig = plt.figure(figsize=(11, 9))
capacity_by_tech.plot(kind='pie', labels=None, startangle=140,
                      autopct=lambda p: f'{p:.1f}%' if p >= 2 else '', pctdistance=0.8,
                      colors=[colors[t] for t in capacity_by_tech.index], wedgeprops={'edgecolor': 'black'},
                      textprops={'fontsize': 14})
plt.title("Current Share of Renewable Energy Capacity by Technology Type", fontsize=18); plt.ylabel("")
pct = capacity_by_tech / capacity_by_tech.sum() * 100
plt.legend([f"{t} - {p:.1f}%" for t, p in pct.items()], title="Technology Type", loc='upper left',
           bbox_to_anchor=(1, 1), fontsize=13, title_fontsize=14)
plt.tight_layout(); plt.savefig("figures/rq1_capacity_share_pie.png", dpi=150, bbox_inches="tight"); plt.show()

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\4122525614.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figures/rq1_capacity_share_pie.png", dpi=150, bbox_inches="tight"); plt.show()


Wind accounts for roughly two-thirds of operational capacity (onshore + offshore), with solar close to a quarter. Everything else is small.

*How the technology mix has shifted over time - by project count and by capacity.*

In [11]:
def stacked_area(values, ylab, title, fname, legend=True):
    cum = (values.unstack(fill_value=0).reindex(columns=tech_types, fill_value=0).cumsum())
    prop = cum.div(cum.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(13, 6.5))
    prop.plot(kind="area", stacked=True, color=[colors[t] for t in tech_types], ax=ax, linewidth=0, legend=False)
    ax.set_ylabel(ylab, fontsize=15); ax.set_xlabel("Year", fontsize=15); ax.set_title(title, fontsize=16)
    ax.tick_params(labelsize=13)
    if legend:
        ax.legend(title="Technology Type", loc="upper left", bbox_to_anchor=(1, 1), fontsize=12, title_fontsize=13)
    ax.set_xlim(1990, prop.index.max()); ax.set_ylim(0, 1)
    plt.tight_layout(); plt.savefig(f"figures/{fname}", dpi=150, bbox_inches="tight"); plt.show()

stacked_area(operational_data.groupby(["Year", "Technology Type"]).size(),
             "Proportion of Operational Projects", "Stacked Proportion of Operational Projects by Technology Type",
             "rq1_project_count_stacked_area.png")
# the capacity area chart sits beside the pie in the report, which already carries the shared
# technology legend, so we drop this one's legend to avoid showing the same key twice.
stacked_area(operational_data.groupby(["Year", "Technology Type"])["Installed Capacity (MW)"].sum(),
             "Proportion of Installed Capacity (MW)", "Stacked Proportion of Installed Capacity by Technology Type",
             "rq1_capacity_stacked_area.png", legend=False)

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\4254524796.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"figures/{fname}", dpi=150, bbox_inches="tight"); plt.show()


C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\4254524796.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"figures/{fname}", dpi=150, bbox_inches="tight"); plt.show()


The story runs from an early waste-to-energy boom (late 1990s), through the mid-2000s rise of onshore wind and the early-2010s solar surge, to the steady climb of offshore wind. By *capacity*, wind dominates: individual wind projects are far larger than the numerous small solar sites.

## 4 | RQ2 - Regional distribution over time

Each operational project carries a commissioning date, so we can map **cumulative capacity by region** at successive points in time. One important caveat drives the design: **offshore wind (~a third of capacity) is not a land region**, so it cannot sit on a NUTS1 choropleth. Rather than silently drop it, we map the land regions and show offshore in a companion panel.

*Static 2x2 snapshots (2010/2015/2020/2025) with a shared colour scale, plus an offshore side-panel on the same scale.*

In [12]:
import geopandas as gpd
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

region_mapping = {
    "North East": "North East (England)", "North West": "North West (England)",
    "Yorkshire and Humber": "Yorkshire and The Humber", "East Midlands": "East Midlands (England)",
    "West Midlands": "West Midlands (England)", "Eastern": "East of England", "London": "London",
    "South East": "South East (England)", "South West": "South West (England)",
    "Scotland ": "Scotland", "Northern Ireland ": "Northern Ireland", "Wales": "Wales"}

op = data[data["Development Status"] == "Operational"].dropna(subset=["Operational"]).copy()
op["Year"] = op["Operational"].dt.year.astype(int)
op["Region"] = op["Region"].replace(region_mapping)
offshore = op[op["Region"] == "Offshore"]; land = op[op["Region"] != "Offshore"]
gdf = gpd.read_file("data/NUTS1_Jan_2018_SGCB_in_the_UK_2022_7531557960096889953.geojson")

SNAPS = [2010, 2015, 2020, 2025]; cmap = "viridis"
cum_land = lambda y: land[land["Year"] <= y].groupby("Region")["Installed Capacity (MW)"].sum() / 1000.0
cum_off = lambda y: offshore[offshore["Year"] <= y]["Installed Capacity (MW)"].sum() / 1000.0
# the shared colour scale must span BOTH the land regions AND offshore (which reaches ~15 GW),
# otherwise offshore is drawn off the top of the scale and the colourbar mis-states its value.
vmax = max(max(cum_land(y).max() for y in SNAPS), max(cum_off(y) for y in SNAPS)); norm = Normalize(0, vmax)

fig = plt.figure(figsize=(15, 11))
gs = GridSpec(2, 3, width_ratios=[1, 1, 0.5], wspace=0.02, hspace=0.20, figure=fig)
maps = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
for ax, y in zip(maps, SNAPS):
    s = cum_land(y)
    g = gdf.merge(s.rename("gw"), left_on="nuts118nm", right_index=True, how="left"); g["gw"] = g["gw"].fillna(0)
    g.plot(column="gw", cmap=cmap, norm=norm, ax=ax, edgecolor="white", linewidth=0.4)
    ax.set_title(f"{y}   -   onshore total {s.sum():.1f} GW", fontsize=13); ax.axis("off")
ax_off = fig.add_subplot(gs[:, 2]); off_vals = [cum_off(y) for y in SNAPS]
bars = ax_off.bar([str(y) for y in SNAPS], off_vals, color=[plt.get_cmap(cmap)(norm(v)) for v in off_vals], edgecolor="black")
ax_off.set_title("Offshore wind\n(shares map colour scale)", fontsize=13); ax_off.set_ylabel("Cumulative capacity (GW)")

# put the offshore y-axis on the right so it doesn't compete with the adjacent map panels
ax_off.yaxis.tick_right(); ax_off.yaxis.set_label_position("right")
ax_off.spines[["top", "left"]].set_visible(False); ax_off.set_ylim(0, max(off_vals) * 1.18)
for b, v in zip(bars, off_vals): ax_off.text(b.get_x() + b.get_width()/2, v + 0.2, f"{v:.1f}", ha="center")
sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])

fig.colorbar(sm, ax=maps, fraction=0.03, pad=0.02, location="bottom", aspect=45).set_label("Cumulative operational capacity by region (GW)")
fig.suptitle("Regional build-out of operational renewable capacity, 2010-2025", fontsize=18, y=0.96)
fig.savefig("figures/rq2_regional_snapshots_2x2.png", dpi=150, bbox_inches="tight"); plt.show()

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\1531310909.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig("figures/rq2_regional_snapshots_2x2.png", dpi=150, bbox_inches="tight"); plt.show()


Scotland dominates onshore capacity throughout, and the South of England fills in with solar later. But the standout is **offshore wind**: from 1.4 GW in 2010 to 15.1 GW in 2025, it now exceeds any single land region - a national story that a regional map alone would miss.

*The same data as an interactive time-slider map (best viewed in a live notebook / the standalone HTML).*

In [13]:
import json
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

years = list(range(2000, int(land["Year"].max()) + 1)); regions = sorted(land["Region"].unique())
def z_for(y):
    s = land[land["Year"] <= y].groupby("Region")["Installed Capacity (MW)"].sum() / 1000.0
    return [round(float(s.get(r, 0.0)), 3) for r in regions]
vmax_i = max(max(z_for(y)) for y in years)

gj = gpd.read_file("data/NUTS1_Jan_2018_SGCB_in_the_UK_2022_7531557960096889953.geojson").to_crs(27700)
gj["geometry"] = gj.geometry.simplify(1000); gj = gj.to_crs(4326)
geojson = json.loads(gj[["nuts118nm", "geometry"]].to_json())

base = go.Choropleth(geojson=geojson, locations=regions, featureidkey="properties.nuts118nm",
                     z=z_for(years[-1]), zmin=0, zmax=vmax_i, colorscale="Viridis", colorbar_title="GW",
                     marker_line_color="white", marker_line_width=0.4,
                     hovertemplate="%{location}<br>%{z:.2f} GW<extra></extra>")
frames = [go.Frame(name=str(y), data=[go.Choropleth(z=z_for(y))]) for y in years]
slider = dict(active=len(years)-1, currentvalue={"prefix": "Year: "}, pad={"t": 40},
              steps=[dict(method="animate", label=str(y),
                          args=[[str(y)], dict(mode="immediate", frame=dict(duration=250, redraw=True), transition=dict(duration=0))]) for y in years])
play = dict(type="buttons", showactive=False, x=0.05, y=0, xanchor="right", yanchor="top",
            buttons=[dict(label="> Play", method="animate", args=[None, dict(frame=dict(duration=350, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                     dict(label="|| Pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])
figi = go.Figure(data=[base], frames=frames)
figi.update_geos(fitbounds="locations", visible=False)
figi.update_layout(title="UK operational renewable capacity by region, cumulative (offshore shown separately)",
                   margin=dict(l=0, r=0, t=55, b=0), sliders=[slider], updatemenus=[play])
figi.write_html("figures/rq2_regional_timeslider.html", include_plotlyjs=True)
figi.show()

## 5 | RQ3 - Government support schemes

The UK has used three main support schemes - the Renewables Obligation (**RO**), Feed-in Tariff (**FiT**) and Contracts for Difference (**CfD**). We look at how scheme membership relates to project scale and technology. For these plots we include **operational and pending** projects, so as not to miss the large CfD projects still under construction.

*Installed capacity by CfD allocation round (wind highlighted), log scale.*

In [14]:
opd = data[data["Development Status"].isin(["Operational", "Pending"])].copy()

# technology order (also used later for the stacked-bar figure) - defined here since
# this figure now runs first
tech_order = [t for t in ["Wind Offshore", "Wind Onshore", "Solar Photovoltaics", "Biomass",
                           "Waste to Energy", "Hydro", "Tidal", "Geothermal", "Other"]
              if t in opd["Technology Type"].unique()]

# chronological CfD round order (None dropped: n=10,840 / 95.4% of opd, same
# justification as the Figure 1 CfD panel - it swamps the chart and isn't the point)
round_order = [r for r in ["Investment Contract", "AR1", "AR2", "AR3", "AR4",
                            "AR5", "AR6", "AR7", "AR7a"] if r in opd["CfD Allocation Round"].unique()]
opd_rounds = opd[(opd["CfD Allocation Round"] != "None") & (opd["Installed Capacity (MW)"] > 0)].copy()


fig, ax = plt.subplots(figsize=(13, 7))

sns.stripplot(data=opd_rounds, x="CfD Allocation Round", y="Installed Capacity (MW)",
              order=round_order, hue="Technology Type", hue_order=tech_order,
              palette=colors, dodge=False, jitter=0.25, alpha=0.75, size=5, ax=ax)

sns.boxplot(data=opd_rounds, x="CfD Allocation Round", y="Installed Capacity (MW)", order=round_order,
            showcaps=True, boxprops=dict(facecolor="black", alpha=0.07, edgecolor="black", linewidth=1.3),
            whiskerprops=dict(color="black", linewidth=1.3), capprops=dict(color="black", linewidth=1.3),
            medianprops=dict(color="#D62839", linewidth=2.2), showfliers=False, ax=ax)

ax.set_yscale("log")
ymin, ymax = ax.get_ylim(); ax.set_ylim(ymin * 0.5, ymax * 2.2)  # headroom for the annotations below

medians = opd_rounds.groupby("CfD Allocation Round")["Installed Capacity (MW)"].median()
counts = opd_rounds["CfD Allocation Round"].value_counts()
ymin, ymax = ax.get_ylim()
for i, rnd in enumerate(round_order):
    # median, above the box
    ax.text(i, ymax * 0.85, f"median\n{medians.get(rnd, 0):.1f} MW", ha="center", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    # n, low in the plot - same convention as the stacked-bar figure's n= labels
    ax.text(i, ymin * 1.15, f"n={int(counts.get(rnd, 0))}", ha="center", va="bottom", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))

ax.set_title("Installed capacity by CfD allocation round", fontsize=18)
ax.set_xlabel("CfD Allocation Round", fontsize=16); ax.set_ylabel("Installed Capacity (MW, log scale)", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend(title="Technology Type", bbox_to_anchor=(1, 1), fontsize=12, title_fontsize=13)
plt.tight_layout(); plt.savefig("figures/rq3_capacity_by_cfd_round.png", dpi=150, bbox_inches="tight"); plt.show()

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\1881169947.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figures/rq3_capacity_by_cfd_round.png", dpi=150, bbox_inches="tight"); plt.show()


Projects inside CfD rounds are far larger than those outside (`None`), and nearly all the very large projects are wind. The CfD scheme is clearly targeted at utility-scale generation.

*Technology mix within each CfD round (with project counts).*

In [15]:
ct = pd.crosstab(opd["CfD Allocation Round"], opd["Technology Type"], normalize='index')
order = [t for t in ["Wind Offshore", "Wind Onshore", "Solar Photovoltaics", "Biomass",
                     "Waste to Energy", "Hydro", "Tidal", "Geothermal", "Other"] if t in ct.columns]
ct = ct[order].dropna(how="all")
fig, ax = plt.subplots(figsize=(13, 7))
ct.plot(kind="bar", stacked=True, color=[colors[t] for t in order], ax=ax, width=0.8)
ax.set_ylabel("Proportion of Projects", fontsize=16); ax.set_xlabel("CfD Allocation Round", fontsize=16)
ax.set_title("Technology mix within each CfD allocation round", fontsize=18)
ax.tick_params(axis="x", labelsize=14, rotation=0); ax.tick_params(axis="y", labelsize=14)
ax.legend(title="Technology Type", bbox_to_anchor=(1, 1), fontsize=13, title_fontsize=14)
rc = opd["CfD Allocation Round"].value_counts()
for i, rnd in enumerate(ct.index):
    ax.text(i, 0.015, f"n={int(rc.get(rnd, 0))}", ha="center", va="bottom", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
plt.tight_layout(); plt.savefig("figures/rq3_cfd_round_vs_tech.png", dpi=150, bbox_inches="tight"); plt.show()

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\1103538520.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figures/rq3_cfd_round_vs_tech.png", dpi=150, bbox_inches="tight"); plt.show()


Early rounds (AR1-AR3) were dominated by offshore wind; the middle rounds shifted heavily to solar; and the newest round (**AR7**, 2025) is back to 100% offshore wind - a clear policy signal.

*Scheme-associated change in typical (median) project size, by technology, versus a time-matched pool of non-scheme projects (heatmap).*

In [16]:
import matplotlib as mpl
from matplotlib.colors import Normalize

# assign each project to the scheme it actually used (logic unchanged)
def assign_scheme(row):
    if row["CfD Allocation Round"] != "None": return "CfD"
    elif row["RO Banding"] > 0: return "RO"
    elif row["FiT Tariff"] > 0: return "FiT"
    return "None"

scheme_df = opd.copy()
scheme_df["Scheme"] = scheme_df.apply(assign_scheme, axis=1)

# --- scheme eligibility windows (Ofgem / gov.uk), used to build a time-matched comparison group ---
# RO has technology-specific early closures (large solar PV 2015, onshore wind 2016); FiT ran
# 2010-2019; CfD opened in 2014 and is still running, so we treat it as open to today.
SCHEME_WINDOWS = {
    "RO":  {"Solar Photovoltaics": ("2002-04-01", "2015-03-31"),
            "Wind Onshore":        ("2002-04-01", "2016-03-31"),
            "_default":            ("2002-04-01", "2017-03-31")},
    "FiT": {"_default": ("2010-04-01", "2019-03-31")},
    "CfD": {"_default": ("2014-01-01", None)}}

def scheme_window(technology, scheme):
    w = SCHEME_WINDOWS[scheme]
    start, end = w.get(technology, w["_default"])
    start = pd.Timestamp(start)
    end = pd.Timestamp(end) if end is not None else pd.Timestamp.today().normalize()
    return start, end

# comparison pool for a (technology, scheme): projects with Scheme == "None" of that technology
# whose Project Date falls inside that scheme's eligibility window -> contemporaries that could
# plausibly have chosen the scheme, rather than the full cross-era pool of non-scheme projects.
none_pool = scheme_df[scheme_df["Scheme"] == "None"]
def comparison_pool(technology, scheme):
    start, end = scheme_window(technology, scheme)
    m = ((none_pool["Technology Type"] == technology)
         & (none_pool["Project Date"] >= start)
         & (none_pool["Project Date"] <= end))
    return none_pool[m]

# --- build the (technology x scheme) matrices of median-capacity difference ---
schemes = ["RO", "FiT", "CfD"]
pct  = np.full((len(tech_types), len(schemes)), np.nan)   # % difference in median capacity
absd = np.full((len(tech_types), len(schemes)), np.nan)   # absolute difference (MW)
ncount = np.zeros((len(tech_types), len(schemes)), dtype=int)
for r, tech in enumerate(tech_types):
    for c, sch in enumerate(schemes):
        ins = scheme_df[(scheme_df["Scheme"] == sch) & (scheme_df["Technology Type"] == tech)]
        out = comparison_pool(tech, sch)
        ncount[r, c] = len(ins)
        # only rate a cell where the scheme actually has projects of this technology (n>=5) and
        # there is a contemporaneous non-scheme pool to compare against; medians (capacity is
        # heavily right-skewed) not means.
        if len(ins) >= 5 and len(out) > 0:
            in_med, out_med = ins["Installed Capacity (MW)"].median(), out["Installed Capacity (MW)"].median()
            absd[r, c] = in_med - out_med
            if out_med > 0:
                pct[r, c] = (in_med - out_med) / out_med * 100

# diverging colour scale centred at 0; cap the range so a couple of huge cells don't wash out
# the rest (the exact value is always printed in the annotation).
finite = pct[np.isfinite(pct)]
vabs = min(float(np.nanmax(np.abs(finite))), 150.0) if finite.size else 100.0
norm = Normalize(-vabs, vabs)
cmap = mpl.colormaps["RdBu_r"].copy(); cmap.set_bad("#e7e7e7")

fig, ax = plt.subplots(figsize=(7.5, 7.8))
im = ax.imshow(np.ma.masked_invalid(pct), cmap=cmap, norm=norm, aspect="auto")
ax.set_xticks(range(len(schemes))); ax.set_xticklabels(schemes, fontsize=15)
ax.set_yticks(range(len(tech_types))); ax.set_yticklabels(tech_types, fontsize=13)
for tick, tech in zip(ax.get_yticklabels(), tech_types):   # colour row labels with the tech map
    tick.set_color(colors.get(tech, "black"))
ax.set_xlabel("Support scheme", fontsize=15)
ax.set_title("Change in typical (median) project size\nvs time-matched non-scheme projects", fontsize=15)

for r in range(len(tech_types)):
    for c in range(len(schemes)):
        if np.isfinite(pct[r, c]):
            val = pct[r, c]
            tcol = "white" if abs(norm(val) - 0.5) > 0.30 else "black"
            ax.text(c, r - 0.13, f"{val:+.0f}%", ha="center", va="center",
                    fontsize=13, fontweight="bold", color=tcol)
            mw = absd[r, c]
            mw_txt = f"{mw:+.0f} MW" if abs(round(mw)) >= 1 else "~0 MW"
            ax.text(c, r + 0.21, mw_txt, ha="center", va="center",
                    fontsize=9.5, color=tcol, alpha=0.85)
        else:
            # greyed cell: either too few in-scheme projects (n<5) or the technology was not a
            # scheme technology at scale -> show why rather than a fabricated near-0 comparison.
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, hatch="///",
                                       edgecolor="#c9c9c9", linewidth=0.0))
            n = ncount[r, c]
            label = f"n={n}\n(too few)" if n > 0 else "n/a"
            ax.text(c, r, label, ha="center", va="center", fontsize=9, color="#5f5f5f")

ax.set_xticks(np.arange(-.5, len(schemes), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(tech_types), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2); ax.tick_params(which="minor", length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Median capacity vs non-scheme (%)", fontsize=12); cb.ax.tick_params(labelsize=11)
plt.tight_layout(); plt.savefig("figures/rq3_capacity_by_scheme.png", dpi=150, bbox_inches="tight"); plt.show()

# underlying numbers for the write-up
print("percentage difference (in-scheme median vs time-matched non-scheme median):")
print(pd.DataFrame(pct, index=tech_types, columns=schemes).round(0))
print("\nn in-scheme projects:")
print(pd.DataFrame(ncount, index=tech_types, columns=schemes))

percentage difference (in-scheme median vs time-matched non-scheme median):
                       RO  FiT     CfD
Biomass               NaN  NaN     NaN
Waste to Energy      43.0  NaN     NaN
Hydro               -16.0  NaN     NaN
Solar Photovoltaics -17.0  NaN  8056.0
Tidal                 NaN  NaN   270.0
Wind Offshore         NaN  NaN   260.0
Wind Onshore        -34.0  NaN   700.0
Geothermal            NaN  NaN     NaN
Other                 NaN  NaN    21.0

n in-scheme projects:
                      RO  FiT  CfD
Biomass                2    0    1
Waste to Energy        9    0    3
Hydro                  5    0    0
Solar Photovoltaics  129    2  357
Tidal                  0    0    6
Wind Offshore          0    0   28
Wind Onshore          24    0   99
Geothermal             0    0    1
Other                  2    0    6


C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\2711790795.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figures/rq3_capacity_by_scheme.png", dpi=150, bbox_inches="tight"); plt.show()


Once each scheme is compared against its **contemporaries** - non-scheme projects of the same technology active in the same eligibility window - the flat "RO -> FiT -> CfD" size gradient breaks apart. **CfD** carries a large, technology-wide size premium (onshore wind roughly +700%, offshore wind +260% but +800 MW in absolute terms, utility-scale solar far above the tiny contemporaneous non-scheme sites). **RO**, by contrast, shows little or even a *negative* premium against its own-era peers (onshore wind about -34%, solar about -17%): the apparent RO premium in a naive all-era comparison was largely an era effect. **FiT** cannot be judged here - the REPD 150 kW floor and the scheme's 5 MW cap leave too few projects (n<5) in every cell.

## 6 | Dashboard

A single at-a-glance summary for a policy / energy-sector audience: headline figures, the capacity mix, the growth trajectory, where capacity sits, and the policy lever.

In [17]:
# =============================================================================
# Summary dashboard: KPI cards + four supporting charts, one figure.
# Designed for at-a-glance reading (report Figure - see fig:dash).
# =============================================================================
from matplotlib.gridspec import GridSpec

# --- Base data slices -------------------------------------------------------
op_d = data[data["Development Status"] == "Operational"].copy()
pend = data[data["Development Status"] == "Pending"]

# --- Headline KPI numbers ----------------------------------------------------
total_gw = op_d["Installed Capacity (MW)"].sum() / 1000
n_operational_projects = len(op_d)
pipeline_gw = pend["Installed Capacity (MW)"].sum() / 1000

capacity_by_tech = op_d.groupby("Technology Type")["Installed Capacity (MW)"].sum()
wind_share_pct = 100 * capacity_by_tech[["Wind Onshore", "Wind Offshore"]].sum() / capacity_by_tech.sum()

kpi_cards = [
    (f"{total_gw:,.1f} GW",           "Operational capacity",  "#2878B5"),
    (f"{n_operational_projects:,}",   "Operational projects",  "#43AA8B"),
    (f"{wind_share_pct:.0f}%",        "Capacity from wind",    "#74A9CF"),
    (f"{pipeline_gw:,.0f} GW",        "Pending pipeline",      "#F3722C"),
]

# --- Figure + grid layout ----------------------------------------------------
# Row 0: four KPI cards. Row 1: tech mix + growth curve. Row 2: regions + scheme.
plt.rcParams.update({"font.size": 11})
fig = plt.figure(figsize=(16, 11))
gs = GridSpec(
    3, 4,
    height_ratios=[0.42, 1, 1],
    hspace=0.42, wspace=0.32,
    left=0.06, right=0.97, top=0.88, bottom=0.07,
    figure=fig,
)

fig.suptitle("UK Renewable Energy - Operational Fleet at a Glance",
             fontsize=22, fontweight="bold", x=0.06, ha="left", y=0.965)
fig.text(0.06, 0.915,
         "Where the UK's operational renewable capacity sits today, how it grew, and what drove it",
         fontsize=12.5, ha="left", color="#444444")
fig.text(0.97, 0.02,
         "Source: DESNZ Renewable Energy Planning Database, Q1 2026 (operational projects). "
         "Offshore wind reported separately.",
         fontsize=8.5, ha="right", color="#777777")

# --- Row 0: KPI cards ---------------------------------------------------------
for i, (headline, label, colour) in enumerate(kpi_cards):
    ax = fig.add_subplot(gs[0, i])
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                facecolor=colour, alpha=0.12, edgecolor=colour, linewidth=1.5))
    ax.text(0.5, 0.60, headline, ha="center", va="center", fontsize=27, fontweight="bold", color=colour)
    ax.text(0.5, 0.20, label, ha="center", va="center", fontsize=12, color="#333333")

# --- Panel A (row 1, left): capacity share by technology ---------------------
ax_tech_mix = fig.add_subplot(gs[1, 0:2])
tech_share_pct = (capacity_by_tech / capacity_by_tech.sum() * 100).sort_values()

ax_tech_mix.barh(tech_share_pct.index, tech_share_pct.values,
                  color=[colors[t] for t in tech_share_pct.index], edgecolor="white")
for y, v in enumerate(tech_share_pct.values):
    ax_tech_mix.text(v + 0.4, y, f"{v:.1f}%", va="center", fontsize=9)

ax_tech_mix.set_title("Capacity mix by technology", fontsize=14, loc="left", fontweight="bold")
ax_tech_mix.set_xlabel("Share of operational capacity (%)")
ax_tech_mix.set_xlim(0, tech_share_pct.max() * 1.15)
ax_tech_mix.spines[["top", "right"]].set_visible(False)

# --- Panel B (row 1, right): cumulative capacity growth over time ------------
ax_growth = fig.add_subplot(gs[1, 2:4])
op_by_year = op_d.dropna(subset=["Operational"]).copy()
op_by_year["Year"] = op_by_year["Operational"].dt.year

cumulative_gw = (op_by_year.groupby("Year")["Installed Capacity (MW)"].sum().cumsum() / 1000)
cumulative_gw = cumulative_gw[cumulative_gw.index >= 2000]

ax_growth.fill_between(cumulative_gw.index, cumulative_gw.values, color="#2878B5", alpha=0.25)
ax_growth.plot(cumulative_gw.index, cumulative_gw.values, color="#2878B5", linewidth=2.5)
ax_growth.annotate(f"{cumulative_gw.iloc[-1]:.0f} GW",
                    xy=(cumulative_gw.index[-1], cumulative_gw.iloc[-1]),
                    xytext=(-6, -18), textcoords="offset points", fontweight="bold", color="#2878B5")

ax_growth.set_title("Cumulative operational capacity, 2000-2026", fontsize=14, loc="left", fontweight="bold")
ax_growth.set_ylabel("GW")
ax_growth.set_xlim(2000, cumulative_gw.index.max())
ax_growth.spines[["top", "right"]].set_visible(False)

# --- Panel C (row 2, left): capacity by region + offshore ---------------------
ax_regions = fig.add_subplot(gs[2, 0:2])
op_by_region = op_d.copy()
op_by_region["Region"] = op_by_region["Region"].replace(region_mapping)

capacity_by_region = op_by_region.groupby("Region")["Installed Capacity (MW)"].sum().sort_values() / 1000
bar_colours = ["#2878B5" if region == "Offshore" else "#9DB7CD" for region in capacity_by_region.index]

ax_regions.barh(capacity_by_region.index, capacity_by_region.values, color=bar_colours, edgecolor="white")
offshore_row = list(capacity_by_region.index).index("Offshore")
ax_regions.text(capacity_by_region["Offshore"] - 0.3, offshore_row, "Offshore",
                 va="center", ha="right", color="white", fontsize=9, fontweight="bold")

ax_regions.set_title("Where capacity sits - regions & offshore", fontsize=14, loc="left", fontweight="bold")
ax_regions.set_xlabel("Operational capacity (GW)")
ax_regions.spines[["top", "right"]].set_visible(False)

# --- Panel D (row 2, right): median project size by support scheme -----------
ax_scheme = fig.add_subplot(gs[2, 2:4])
scheme_df = data[data["Development Status"].isin(["Operational", "Pending"])].copy()
scheme_df["Scheme"] = scheme_df.apply(assign_scheme, axis=1)

median_by_scheme = (
    scheme_df[scheme_df["Scheme"].isin(["RO", "FiT", "CfD"])]
    .groupby("Scheme")["Installed Capacity (MW)"]
    .median()
    .reindex(["RO", "FiT", "CfD"])
)

ax_scheme.bar(median_by_scheme.index, median_by_scheme.values,
              color=["#BEBEBE", "#74A9CF", "#2878B5"], edgecolor="black")
for x, v in enumerate(median_by_scheme.values):
    ax_scheme.text(x, v + 0.8, f"{v:.0f} MW", ha="center", fontsize=10, fontweight="bold")

ax_scheme.set_title("Bigger projects use CfD support", fontsize=14, loc="left", fontweight="bold")
ax_scheme.set_ylabel("Median project size (MW)")
ax_scheme.set_ylim(0, median_by_scheme.max() * 1.25)
ax_scheme.spines[["top", "right"]].set_visible(False)

fig.savefig("figures/dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\wsidn\AppData\Local\Temp\ipykernel_23380\1804555515.py:130: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 | Conclusion

Across the three questions a consistent picture emerges. **By capacity, wind is king** (~~two-thirds of the operational fleet), with solar a strong second - a mix built up over three decades of policy-driven waves. **Geographically**, Scotland leads onshore while offshore wind, sitting outside regional boundaries, has become the single largest source and the main growth engine. And the **support schemes** map cleanly onto project scale: the CfD scheme underwrites the utility-scale wind that dominates recent capacity growth. Full interpretation, limitations and links to the literature are developed in the accompanying report.

## References

*See the accompanying report for the full, referenced discussion.*